# HOLE2：在 Google Colab 中分析 PDB

使用普通 **CPU** 运行时，按顺序运行下面的单元格。此入口直接安装 HOLE 2.3.1 并复用项目计算模块，不启动 Docker 或额外的 Web 服务。

数据在 `/content/hole2-data` 中，运行时回收后会丢失。请下载结果，或运行最后的 Google Drive 保存单元格。ZIP 在绘图前创建，不包含随后生成的图片；Drive 导出会同时保存原始 ZIP 和单独的图片。

首次安装需要网络连接。重新连接一个全新运行时后，需要重新执行安装和上传。

## 1. 准备代码和 HOLE 工具

保留 Colab 自己的 Python 内核，仅为 HOLE 创建独立的原生工具环境。重复执行此单元格会复用已安装的工具，不删除工作数据。若需获取项目更新，请使用一个全新运行时重新打开本 Notebook。

In [ ]:
from pathlib import Path
import subprocess
import sys

SOURCE = Path('/content/hole2-colab-source')
if not SOURCE.exists():
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/guoyaobit/hole2-quickstart.git', str(SOURCE)],
        check=True, timeout=180,
    )
if not (SOURCE / 'colab_support.py').is_file():
    raise RuntimeError('代码目录不完整。请断开并删除运行时，然后重新运行安装单元格。')
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

from colab_support import install_hole, import_pdbs, run_batch, save_results

prefix = install_hole()
WORKSPACE = Path('/content/hole2-data')
WORKSPACE.mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
print('代码提交:', commit)
print('HOLE 环境:', prefix)
print('工作目录:', WORKSPACE)


## 2. 上传 PDB

选择一个或多个 `.pdb` / `.PDB` 文件。输入文件名会转换为适合 HOLE 的简单英文文件名；重名时添加后缀，保留此前上传的文件。后续参数请使用输出中列出的文件名。

In [ ]:
from google.colab import files
import tempfile

with tempfile.TemporaryDirectory() as staging:
    uploaded = files.upload(target_dir=staging)
    imported = import_pdbs(uploaded.keys(), WORKSPACE)
print('本次导入:', imported)
print('当前所有 PDB:', sorted(p.name for p in WORKSPACE.glob('*.pdb') if p.is_file()))


### 可选：从 Google Drive 导入

如果已上传本地文件，可跳过本单元格。启用后需要你在 Colab 中授权挂载 Drive；原始 Drive 文件只被复制到本地工作目录，计算在本地磁盘进行。

In [ ]:
use_drive_input = False  #@param {type:"boolean"}
drive_input_folder = '/content/drive/MyDrive/hole2-input'  #@param {type:"string"}

if use_drive_input:
    from google.colab import drive
    drive.mount('/content/drive')
    source_folder = Path(drive_input_folder)
    source_files = sorted(p for p in source_folder.iterdir() if p.is_file() and p.suffix.lower() == '.pdb')
    if not source_files:
        raise ValueError('指定的 Drive 文件夹中没有 PDB 文件。')
    print('已导入:', import_pdbs(source_files, WORKSPACE))
else:
    print('已跳过 Drive 导入。')


## 3. 设置每个文件的参数

修改下面的 `parameters` 字典，键使用第 2 步显示的工作文件名。每个向量必须有三个有限数字，`CVECT` 不能全为零。不填写的文件使用 HOLE 默认值。

`selected_files` 默认包含工作目录中的全部 PDB；可以改成指定列表，例如 `['sample.pdb']`。每次重新执行此单元格，会按这里的代码重置参数。

In [ ]:
parameters = {
    # 'sample.pdb': {'CPOINT': [124.459, 125.13, 127.348], 'CVECT': [0, 0, 1]},
}
selected_files = sorted(p.name for p in WORKSPACE.glob('*.pdb') if p.is_file())
# selected_files = ['sample.pdb']

print('选择的文件:', selected_files)
print('指定的参数:', parameters)


## 4. 计算并显示曲线

按文件顺序处理，避免免费运行时并发占用过多资源。已存在的结果目录会保留。全部文件计算完成后先生成原始结果 ZIP，再绘图。若只有绘图失败，仍可下载已经生成的原始 ZIP。

In [ ]:
from IPython.display import display, Image

result = None
result = run_batch(WORKSPACE, parameters=parameters, selected=selected_files)
print('原始结果 ZIP:', result['archive'])
for plot in result['plots']:
    if plot.lower().endswith('.png'):
        display(Image(filename=plot))
    else:
        print('绘图数据 CSV:', plot)
        print(Path(plot).read_text()[:2000])
for folder, error in result['plot_errors'].items():
    print('绘图警告:', folder, error)


## 5. 下载原始 ZIP

ZIP 不包含第 4 步之后生成的图片。图片可从左侧文件面板 `/content/hole2-data` 的各结果目录单独下载，或与 ZIP 一起保存到 Google Drive。

In [ ]:
if globals().get('result') is None:
    raise RuntimeError('请先成功运行第 4 步。')
from google.colab import files
files.download(result['archive'])


## 6. 可选：将 ZIP 和图片保存到 Google Drive

启用后需要挂载授权。每次导出创建一个新的子目录，保留已有导出。这里只保存结果，不会自动保存第 3 步的内存状态；如需复用参数，请保存 Notebook 的副本。

In [ ]:
save_to_drive = False  #@param {type:"boolean"}
drive_output_folder = '/content/drive/MyDrive/hole2-results'  #@param {type:"string"}

if save_to_drive:
    if globals().get('result') is None:
        raise RuntimeError('请先成功运行第 4 步。')
    from google.colab import drive
    drive.mount('/content/drive')
    saved = save_results(result, drive_output_folder)
    print('已保存 ZIP 和单独的图片/CSV:', saved)
else:
    print('已跳过 Drive 保存。请确保已经下载需要的结果。')
